# 08 · The grid (stage S7)

Every item at every kept fraction `k` and loop count `R`, with the compressor chosen in S5, controller features on.
The controller later learns and is evaluated from this table (`grid.parquet`) without running the model again.
Items: `ctrl_train` + `dev` (the test split stays sealed until S12). A sample of `n_rerun` items runs a second time
(`rerun_id = 1`) to measure how much the results move on their own.

**How to run:** Run All, no edits: one item, then a 20-item sample. `RUN_FULL` is off; read the sample first.
The full run is long and resumable. `compressor` must be frozen in `configs/mvp.yaml` for the full run.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_for_notebook

CONFIG = "configs/mvp.yaml"
GPU_IDS, N_COPIES = select_for_notebook(CONFIG)

In [ ]:
import math
import os
import socket

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/mvp.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())

In [ ]:
import torch

IS_INARA = "inara" in socket.gethostname()
print("host", socket.gethostname(), "| on Inara:", IS_INARA)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd

In [ ]:
# Settings
RUN_FULL = False  # set True after the sample table is read
N_SAMPLE = cfg["sample_items"]  # step 3: dev items, half GQA, half TextVQA
FROZEN = cfg["compressor"] != "none"
grid_cfg = cfg if FROZEN else {**cfg, "compressor": "text_mean"}  # S5 not frozen: sample runs use text_mean
OUT_DIR = ROOT / cfg["out_dir"] / "grid"
SAMPLE_DIR = ROOT / cfg["out_dir"] / "grid_sample"
CELLS = [(k, R) for k in cfg["k_grid"] for R in cfg["R_grid"]]
print("compressor", grid_cfg["compressor"], "(frozen)" if FROZEN else "(NOT frozen: sample only)", "| cells", len(CELLS))
cfg["grid"], cfg["n_rerun"]

### Step-1:- Load the model

In [ ]:
print("[STAGE] load model")
vlms = api.load_models(cfg, N_COPIES)
vlm = vlms[0]

### Step-2:- One item per benchmark, every cell (paste this output)

Raw output, parsed answer and references at R = 3 for each k, then the row of every cell and its features.

In [ ]:
one_items = [api.load_gqa(cfg, "dev", limit=1, root=ROOT)[0], api.load_textvqa(cfg, "dev", limit=1, root=ROOT)[0]]
print("[STAGE] one item per benchmark")
one = {}
for it in one_items:
    rows, features = api.run_item_cells(vlm, it, grid_cfg)
    one[it.benchmark] = (rows, features)
    for row in rows:
        if row["R"] == 3:
            print(f"{it.benchmark} | k={row['k']} R=3 | kept {row['n_vis']}/{row['n_vis_full']}")
            print(f"    raw {row['raw_output']!r} | pred {row['pred']!r} | gold {it.answers[:3]}")
pd.DataFrame([r for rows, _ in one.values() for r in rows])[
    ["benchmark", "k", "R", "n_vis", "n_input", "pred", "correct", "ttft_s", "generate_s"]]

In [ ]:
for benchmark, (rows, features) in one.items():
    assert sorted((r["k"], r["R"]) for r in rows) == sorted(CELLS) == sorted(features), benchmark
    assert all(r["parse_ok"] for r in rows), f"{benchmark}: an answer did not parse"
    assert all(torch.isfinite(f["h_last"]).all() for f in features.values())
print(f"✓ one item per benchmark: {len(CELLS)} cells each, answers parse, features finite")

#### Batched against batch 1, with features

The grid runs equal-length batches of at most `eval_batch_size` items. Check: 16 dev items per benchmark at k = 25% and
100%, batched and each alone. The answers must be identical. Features come from floating-point states, so they are
compared with a tolerance. If it fails, keep `eval_batch_size: 1` and report.

In [ ]:
CHECK_CFG = {**grid_cfg, "k_grid": [0.25, 1.0], "R_grid": [1, 2, 3]}  # for this check only
N_BATCH_CHECK = 16  # items per benchmark
CHECK_BATCH = 8  # batch size under test, whatever eval_batch_size is
check_items = []
for load in (api.load_gqa, api.load_textvqa):
    check_items += api.batchable_items(vlm, load(cfg, "dev", root=ROOT), CHECK_CFG, N_BATCH_CHECK)
print("[STAGE] equal-length batches against batch 1")
compare_rows = []
for chunk in api.item_chunks(vlm, check_items, CHECK_CFG, CHECK_BATCH):
    together = api.run_items_cells(vlm, chunk, CHECK_CFG)
    alone = [api.run_item_cells(vlm, it, CHECK_CFG) for it in chunk]
    for it, (rows_b, feats_b), (rows_a, feats_a) in zip(chunk, together, alone):
        for row_b, row_a in zip(rows_b, rows_a):
            key = row_a["k"], row_a["R"]
            fa, fb = feats_a[key], feats_b[key]
            compare_rows.append({
                "item_id": it.item_id, "batch_n": len(chunk), "k": row_a["k"], "R": row_a["R"],
                "same_output": row_a["raw_output"] == row_b["raw_output"], "same_pred": row_a["pred"] == row_b["pred"],
                "h_cosine": torch.cosine_similarity(fa["h_last"].float(), fb["h_last"].float(), dim=0).item(),
                "entropy_diff": abs(fa["entropy"] - fb["entropy"]),
                "vis_mass_diff": abs(fa["vis_attn_mass"] - fb["vis_attn_mass"])})
batch_check = pd.DataFrame(compare_rows)
batch_check[["batch_n", "same_output", "same_pred", "h_cosine", "entropy_diff", "vis_mass_diff"]].agg(["min", "max"])

In [ ]:
batched_rows = batch_check[batch_check.batch_n > 1]
assert len(batched_rows) >= len(batch_check) // 2, "too few rows were really batched to prove anything"
different = batched_rows[~(batched_rows.same_output & batched_rows.same_pred)]
assert different.empty, f"batching changed {len(different)} outputs:\n{different.to_string()}"
assert batched_rows.h_cosine.min() > 0.999, f"batched h_last differs: min cosine {batched_rows.h_cosine.min()}"
print(f"✓ equal-length batches = batch 1: same outputs on {len(batched_rows)} batched cells; "
      f"h_last cosine >= {batched_rows.h_cosine.min():.5f}, largest entropy gap {batched_rows.entropy_diff.max():.4f}")

### Step-3:- A sample through the grid runner

The same function as the full run, on `N_SAMPLE` dev items, with the re-run of 2 of them, merged to a parquet file.

In [ ]:
print("[STAGE] sample grid")
sample = api.load_gqa(cfg, "dev", limit=N_SAMPLE // 2, root=ROOT) + api.load_textvqa(cfg, "dev", limit=N_SAMPLE // 2, root=ROOT)
sample_again = api.rerun_items(sample, 2, cfg["splits"]["seed"])
api.run_grid(vlms, sample, grid_cfg, SAMPLE_DIR)
api.run_grid(vlms, sample_again, grid_cfg, SAMPLE_DIR, rerun_id=1)
sample_grid = api.merge_grid(SAMPLE_DIR, grid_cfg)
sample_grid.head()

In [ ]:
sample_grid[sample_grid.rerun_id == 0].pivot_table(index="k", columns="R", values="correct", aggfunc="mean").mul(100).round(1)

In [ ]:
sample_grid[sample_grid.rerun_id == 0].pivot_table(index="k", columns="R", values="flops_total", aggfunc="mean").div(1e9).round(2)

In [ ]:
main, again = sample_grid[sample_grid.rerun_id == 0], sample_grid[sample_grid.rerun_id == 1]
assert len(main) == len(sample) * len(CELLS) and len(again) == len(sample_again) * len(CELLS), (len(main), len(again))
assert main.groupby(["benchmark", "item_id"]).size().eq(len(CELLS)).all()
feature_dir = SAMPLE_DIR / "features" / "base" / api.config_hash(grid_cfg)
assert all((feature_dir / f"{it.benchmark}_{it.item_id}.pt").is_file() for it in sample)
pair = main.merge(again, on=["benchmark", "item_id", "k", "R"], suffixes=("", "_again"))
print(f"re-run: {int((pair.pred.fillna('') != pair.pred_again.fillna('')).sum())} of {len(pair)} answers differ")
print(f"✓ sample grid: {len(main)} rows + {len(again)} re-run rows, one features file per item")

### Step-4:- Full run

`RUN_FULL = True` runs every item of the grid splits, then the re-run sample, over the copies of this kernel, and merges
`grid.parquet`. Stop it any time; running the cell again continues. The same run from a terminal, one shard per GPU group
(free this kernel's models first):

```
python scripts/mvp.py grid --shard 0/2
python scripts/mvp.py grid --shard 1/2
```

In [ ]:
items = api.grid_items(grid_cfg, ROOT)
again_items = api.rerun_items(items, cfg["n_rerun"], cfg["splits"]["seed"])
print(len(items), "items +", len(again_items), "re-run | rows expected", (len(items) + len(again_items)) * len(CELLS))

In [ ]:
if RUN_FULL:
    assert FROZEN, "freeze `compressor` in configs/mvp.yaml (S5) before the full run"
    print("[STAGE] full grid")
    api.run_grid(vlms, items, grid_cfg, OUT_DIR)
    api.run_grid(vlms, again_items, grid_cfg, OUT_DIR, rerun_id=1)
else:
    print("skipped: RUN_FULL is False")

#### Merge and count (also after a terminal run)

In [ ]:
has_results = any(OUT_DIR.glob(f"grid_*_{api.config_hash(grid_cfg)}.jsonl"))
grid = api.merge_grid(OUT_DIR, grid_cfg) if has_results else None
grid.groupby(["split", "benchmark", "rerun_id"]).size() if has_results else "no full-run files yet"

In [ ]:
if has_results:
    expected = {0: len(items) * len(CELLS), 1: len(again_items) * len(CELLS)}
    got = grid.rerun_id.value_counts().to_dict()
    assert got == expected, f"grid incomplete: rows {got}, expected {expected}"
    assert not (grid.split == "test").any()
    print(f"✓ grid.parquet complete: {len(grid)} rows in {OUT_DIR}")
else:
    print("skipped: no full-run files yet")